# Create DHSC Awards from NIHR Open Data (DHSC-direct programmes)

Creates awards for the **UK Department of Health and Social Care (DHSC)** from NIHR's
Opendatasoft portal. These are the DHSC-named-funder slices of the NIHR portfolio:

- **Policy Research Programme** rows from `nihr-summary-view` (DHSC commissions PRP directly)
- **NIHR (ODA)** rows (Global Health Research funded from DHSC's UK-aid allocation)
- **Policy Research Unit (PRU) sub-projects** from the dedicated `prp_dataset`

**Attribution care (runbook 2.3.2):** NIHR itself (F4320319990) has a Complete
first-party ingest (provenance `nihr`, priority 13). As of the coordinator-approved
2026-07-12 refresh, `nihr_to_s3.py` DROPS the DHSC-direct rows (Policy Research
Programme / NIHR (ODA)) from the NIHR side, and ALL of them ship here instead —
1,200 rows total (739 summary-view + 461 PRU sub-projects). The refreshed NIHR-side
portfolio (10,763 rows) is mirrored with audit flags in
`s3://openalex-ingest/awards/nihr_ods_dhsc/staging/nihr_full_portfolio_staging.parquet`.
Deploy order note: re-run CreateNIHRAwards and run this notebook in the same session
(either order), then one RefreshWorkAwards after both.

**Prerequisites:**
- Run `scripts/local/nihr_ods_dhsc_to_s3.py` to download and upload the data first.

**Data source:** https://nihr.opendatasoft.com (datasets `nihr-summary-view`, `prp_dataset`; ODS export API)
**S3 location:** `s3a://openalex-ingest/awards/nihr_ods_dhsc/nihr_ods_dhsc_projects.parquet`

**DHSC funder:**
- funder_id: 4320319994
- display_name: "Department of Health and Social Care"
- ROR: https://ror.org/03sbpja79
- DOI: 10.13039/501100000276

**Amount/currency:** GBP, implicit (single-country funder; portal column is
`award_amount_from_dh` = award amount from the Department of Health). The
`prp_dataset` PRU records publish **no amounts** (the dataset has no amount field), so
overall amount coverage is ~61.6% while the summary-view subset is ~100% -> documented
**Step 6.7 waiver** for the PRU subset: NULL is the correct value for fields the funder
doesn't publish.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet
CREATE OR REPLACE TABLE openalex.awards.nihr_ods_dhsc_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nihr_ods_dhsc/nihr_ods_dhsc_projects.parquet`;

In [ ]:
%sql
-- Check row count (should be ~1,200)
SELECT COUNT(*) as total_projects FROM openalex.awards.nihr_ods_dhsc_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.nihr_ods_dhsc_raw LIMIT 5;

In [ ]:
%sql
-- Check column names before writing transformation SQL (runbook 2.2 step 1.5)
DESCRIBE openalex.awards.nihr_ods_dhsc_raw;

## Step 1.6: Funder existence check (Path A — F4320* funder, must return exactly 1 row)

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320319994;  -- Department of Health and Social Care

## Step 2: Create DHSC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nihr_ods_dhsc_awards
USING delta
AS
WITH
-- Get DHSC funder from OpenAlex by explicit funder_id (Path A: F4320* is in the dim)
dhsc_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319994  -- Department of Health and Social Care
),

awards_transformed AS (
    SELECT
        -- Generate unique ID using xxhash64 of funder_id:project_id
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_id)))) % 9000000000 as id,

        -- Display name = project title
        g.project_title as display_name,

        -- Description = plain English abstract (preferred) or scientific abstract
        COALESCE(g.plain_english_abstract, g.scientific_abstract) as description,

        -- Funder info
        f.funder_id,
        g.project_id as funder_award_id,

        -- Amount in GBP (award_amount_from_dh; NULL for prp_dataset PRU rows --
        -- that dataset publishes no amounts; Step 6.7 waiver documented in header)
        TRY_CAST(g.award_amount_from_dh AS DOUBLE) as amount,
        'GBP' as currency,

        -- Funder struct
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type
        CASE
            WHEN LOWER(COALESCE(g.project_title, '')) LIKE '%training award%' THEN 'training'
            WHEN LOWER(COALESCE(g.programme, '')) LIKE '%fellowship%' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,

        -- Funder scheme = programme name (PRU rows carry their portfolio label too)
        COALESCE(g.programme, g.funding_stream) as funder_scheme,

        -- Provenance
        'nihr_ods_dhsc' as provenance,

        -- Dates (stored as strings in YYYY-MM-DD format)
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,

        -- Lead investigator: chief investigator when named; when only the
        -- contracted organisation is known, ship an affiliation-only struct
        CASE
            WHEN g.lead_family_name IS NOT NULL THEN
                struct(
                    g.lead_given_name as given_name,
                    g.lead_family_name as family_name,
                    CASE
                        WHEN g.orcid IS NOT NULL AND g.orcid RLIKE '^[0-9]{4}-'
                            THEN CONCAT('https://orcid.org/', g.orcid)
                        ELSE g.orcid
                    END as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.contracted_organisation as name,
                        'GB' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            WHEN g.contracted_organisation IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.contracted_organisation as name,
                        'GB' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        -- Co-lead and other investigators (not available in the source)
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page URL (NIHR Funding & Awards page; NULL for PRU sub-projects)
        g.funding_and_awards_link as landing_page_url,

        -- No DOI for DHSC awards
        CAST(NULL AS STRING) as doi,

        -- Works API URL
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_id)))) % 9000000000) as works_api_url,

        -- Timestamps
        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.nihr_ods_dhsc_raw g
    CROSS JOIN dhsc_funder f
    WHERE g.project_id IS NOT NULL
      AND TRIM(g.project_id) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nihr_ods_dhsc' AND priority = 425;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    425 as priority  -- DHSC priority
FROM openalex.awards.nihr_ods_dhsc_awards;

## Verification Queries

In [ ]:
%sql
-- 6.1 Basic count (should match the parquet row count, ~1,200)
SELECT COUNT(*) as total_dhsc_awards FROM openalex.awards.nihr_ods_dhsc_awards;

In [ ]:
%sql
-- 6.2 Schema validation
DESCRIBE openalex.awards.nihr_ods_dhsc_awards;

In [ ]:
%sql
-- 6.3 Data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_dates
FROM openalex.awards.nihr_ods_dhsc_awards;

In [ ]:
%sql
-- 6.4 Sample inspection
SELECT id, display_name, funder_award_id, funder_scheme, funding_type,
       amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name as org
FROM openalex.awards.nihr_ods_dhsc_awards LIMIT 10;

In [ ]:
%sql
-- 6.4a PI / display_name frequency check (catches systematic scraper bugs).
-- NOTE: repeated PIs up to ~9 rows are expected here — PRU units re-commission
-- the same chief investigator across multiple sub-project references.
SELECT lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       COUNT(*) AS n
FROM openalex.awards.nihr_ods_dhsc_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.5 Funder consistency (single funder expected: DHSC)
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.nihr_ods_dhsc_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- 6.6 Year distribution
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.nihr_ods_dhsc_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;

In [ ]:
%sql
-- 6.7 Amount and currency coverage.
-- Expected: ~61.6% overall (PRU sub-projects publish no amounts -> waiver, see
-- header) but ~100% on the nihr-summary-view subset. Split by source dataset:
SELECT r.source_dataset,
       COUNT(*) AS total,
       COUNT(a.amount) AS has_amount,
       ROUND(COUNT(a.amount) * 100.0 / COUNT(*), 1) AS pct_amount,
       MIN(a.amount) AS min_amount, MAX(a.amount) AS max_amount,
       ROUND(AVG(a.amount), 0) AS avg_amount
FROM openalex.awards.nihr_ods_dhsc_awards a
JOIN openalex.awards.nihr_ods_dhsc_raw r ON r.project_id = a.funder_award_id
GROUP BY r.source_dataset;

In [ ]:
%sql
-- 6.8 Confirm the rows reached the shared raw table
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nihr_ods_dhsc'
GROUP BY provenance, priority;